Import Modules

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline

import datetime as dt

import time
import os
from os import listdir
from os.path import isfile, join

Part 1: Data Loading and Cleaning

In [ ]:
def save_to_csv_from_yahoo(folder, ticker, syear, smonth, sday, eyear, emonth, eday):
  start=dt.datetime(syear, smonth, sday)
  end=dt.datetime(eyear, emonth, eday)
  # Add .AX suffix for Australian stocks if not already present
  if not ticker.endswith('.AX'):
    ticker_with_suffix = ticker + '.AX'
  else:
    ticker_with_suffix = ticker

  try:
    print("Get Data for :", ticker_with_suffix)
    df=yf.download(ticker_with_suffix, start=start, end=end)
    time.sleep(1)
    # Ensure the folder exists before saving the file
    if not os.path.exists(folder):
        os.makedirs(folder)
    # Only save if data is returned
    if not df.empty:
        # Explicitly save the index as a column named 'Date'
        df.to_csv(folder+'/'+ticker+'.csv', index_label='Date')
    else:
        raise ValueError(f"No data returned for {ticker_with_suffix}")
  except Exception as ex:
    stocks_not_downloaded.append(ticker)
    print("Couldn't Get Data for: ", ticker_with_suffix, ". Error:", ex)

In [5]:
def get_df_from_csv(folder,ticker):
  full_path = folder + '/' + ticker + '.csv'
  try:
    # Read the CSV directly, using 'Date' as the index and parsing dates
    df = pd.read_csv(full_path, encoding='latin1')
    df = df.drop(index=[0, 1]).reset_index(drop=True)
    df.rename(columns={'Price': 'Date'}, inplace=True)
    df.set_index('Date', inplace=True)
    df.index = pd.to_datetime(df.index, errors='coerce')
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop any rows that might have become all NaN after coercion in numeric_cols subset
    # Only keep rows where at least one of the numeric columns is not NaN
    df.dropna(how='all', subset=[col for col in numeric_cols if col in df.columns], inplace=True)

    return df
  except FileNotFoundError:
    print(f"File '{full_path}' doesn't exist")
    return None
  except Exception as e:
    print(f"Error reading file '{full_path}': {e}")
    return None

In [6]:
def save_df_to_csv(df, ticker):
  df.to_csv(path+ '/'+ ticker+'.csv')

In [8]:
def delete_unnamed_cols(df):
  df=df.loc[:, ~df.columns.str.contains('^Unnamed')]
  return df

In [55]:
path=' ' #Enter your own file path here
syear=2021
smonth=1
sday=3
sdate_str='2021-01-03'
sdate_datetime=dt.datetime(syear, smonth, sday)

eyear=2026
emonth=6
eday=22
edate_str='2026-06-22'
edate_datetime=dt.datetime(eyear, emonth, eday)

risk_free_rate=0.043

stocks_not_downloaded=[]
missing_stocks=[]

In [56]:
file_path = " " #Tickers file path
column_name = "Ticker"

try:
  # Try reading with 'latin1' encoding, a common alternative to 'utf-8'
  df = pd.read_csv(file_path, encoding='latin1')
  tickers = df[column_name]
except FileNotFoundError:
  print(f"File '{file_path}' doesn't exist")
except Exception as e:
  print(f"An error occurred: {e}")

File ' ' doesn't exist


In [ ]:
folder=path
for x in range(len(tickers)):
  save_to_csv_from_yahoo(folder,tickers[x],syear,smonth,sday,eyear,emonth,eday)
print("Finished")

In [ ]:
files=[x for x in listdir(path) if isfile(join(folder, x))]
tickers=[os.path.splitext(x)[0] for x in files]
stock_df=pd.DataFrame(tickers, columns=['Ticker'])

Part 2: Risk and Return Metrics

In [9]:
def get_valid_dates(df, sdate, edate):
    try:
        df.index = pd.to_datetime(df.index)
        sdate = pd.to_datetime(sdate)
        edate = pd.to_datetime(edate)

        sm_df = df.loc[(df.index > sdate) & (df.index <= edate)]

        if sm_df.empty:
            print("No data in date range")
            return None, None

        sm_date = sm_df.index.min().strftime('%Y-%m-%d')
        last_date = sm_df.index.max().strftime('%Y-%m-%d')

    except Exception as e:
        print(f"Date corrupted: {e}")
        return None, None
    else:
        return sm_date, last_date

In [10]:
def add_daily_return_to_df(df, ticker):
  df['daily_return']=(df['Close']/df['Close'].shift(1))-1
  df.to_csv(path+ '/'+ ticker+'.csv')
  return df

In [30]:
def get_mean_between_dates(df, ticker, sdate, edate):
  sdate = pd.to_datetime(sdate)
  edate = pd.to_datetime(edate)
  df.index = pd.to_datetime(df.index)
  mask = (df.index >= sdate) & (df.index <= edate)
  return df.loc[mask]['daily_return'].mean()

In [39]:
def get_sd_between_dates(df, ticker, sdate, edate):
    sdate = pd.to_datetime(sdate)
    edate = pd.to_datetime(edate)
    df.index = pd.to_datetime(df.index)
    mask = (df.index >= sdate) & (df.index <= edate)
    return df.loc[mask]['daily_return'].std()

In [40]:
def get_cov_between_dates(df, ticker, sdate, edate):
  mean=get_mean_between_dates(df, ticker, sdate, edate)
  sd=get_sd_between_dates(df, ticker, sdate,edate)
  return sd/mean

In [16]:
def merge_df_by_column_name(col_name, sdate, edate, *tickers):
    mult_df = pd.DataFrame()
    sdate = pd.to_datetime(sdate)
    edate = pd.to_datetime(edate)
    for x in tickers:
        df = get_df_from_csv(path, x)
        df.index = pd.to_datetime(df.index)
        mask = (df.index >= sdate) & (df.index <= edate)
        mult_df[x] = df.loc[mask][col_name]
    return mult_df

In [41]:
def get_rois_for_stocks(stock_df):
    tickers = []
    means = []
    sds = []
    covs = []
    for index, row in stock_df.iterrows():
        df = get_df_from_csv(path, row['Symbol'])
        if df is None:
            continue
        sdate, edate = get_valid_dates(df, '2021-01-03', '2026-06-22')
        if sdate is None or edate is None:
            continue
        tickers.append(row['Symbol'])
        mean = get_mean_between_dates(df, row['Symbol'], sdate, edate)
        sd = get_sd_between_dates(df, row['Symbol'], sdate, edate)
        cov = get_cov_between_dates(df, row['Symbol'], sdate, edate)
        means.append(mean)
        sds.append(sd)
        covs.append(cov)
    return pd.DataFrame({
        'Ticker': tickers,
        'Mean Return': means,
        'Std Dev': sds,
        'COV': covs
    })

In [ ]:
for ticker in tickers:
    print(f'Working on {ticker}')
    df=get_df_from_csv(path,ticker)
    df=add_daily_return_to_df(df,ticker)
print("Finished")

In [34]:
sec_df=pd.read_csv('C:/mervyn/Python/Portfolio analysis tool/stock_sector.csv')
sectors=set(sec_df["Sector"])

indus_df=sec_df.loc[sec_df['Sector']=='Industrials']
health_df=sec_df.loc[sec_df['Sector']=='Health Care']
it_df=sec_df.loc[sec_df['Sector']=='Information Technology']
energy_df=sec_df.loc[sec_df['Sector']=='Energy']
restate_df=sec_df.loc[sec_df['Sector']=='Real estate']
comm_df=sec_df.loc[sec_df['Sector']=='Communicatio Services']
financial_df=sec_df.loc[sec_df['Sector']=='Financials']
discretion_df=sec_df.loc[sec_df['Sector']=='Consumer Discretionary']
staple_df=sec_df.loc[sec_df['Sector']=='Cosumer Staples']
material_df=sec_df.loc[sec_df['Sector']=='Materials']
utility_df=sec_df.loc[sec_df['Sector']=='Utilities']
etf_df=sec_df.loc[sec_df['Sector']=='ETFs']

In [ ]:
industrial=get_rois_for_stocks(indus_df)
health_care=get_rois_for_stocks(health_df)
it=get_rois_for_stocks(it_df)
energy=get_rois_for_stocks(energy_df)
restate=get_rois_for_stocks(restate_df)
comm=get_rois_for_stocks(comm_df)
finance=get_rois_for_stocks(financial_df)
discretion=get_rois_for_stocks(discretion_df)
staple=get_rois_for_stocks(staple_df)
material=get_rois_for_stocks(material_df)
utility=get_rois_for_stocks(utility_df)
etf=get_rois_for_stocks(etf_df)

In [ ]:
print(industrial.sort_values(by=['Mean Return'], ascending=False).head(5))
print(health_care.sort_values(by=['Mean Return'], ascending=False).head(5))
print(material.sort_values(by=['Mean Return'], ascending=False).head(5))
print(finance.sort_values(by=['Mean Return'], ascending=False).head(5))
print(staple.sort_values(by=['Mean Return'], ascending=False).head(5))
print(it.sort_values(by=['Mean Return'], ascending=False).head(5))
print(energy.sort_values(by=['Mean Return'], ascending=False).head(5))
print(restate.sort_values(by=['Mean Return'], ascending=False).head(5))
print(comm.sort_values(by=['Mean Return'], ascending=False).head(5))
print(discretion.sort_values(by=['Mean Return'], ascending=False).head(5))
print(utility.sort_values(by=['Mean Return'], ascending=False).head(5))
print(etf.sort_values(by=['Mean Return'], ascending=False).head(5))

Part 3: Portfolio Optimization and Visualisation

In [ ]:
port_list=[]
#Add your own portfolio of stocks
#Chosen portfolio "BHP.AX","EVN.AX","ANZ.AX","WOW.AX","TLS.AX","AGL.AX","STO.AX","MPL.AX","QAN.AX"

In [ ]:
mult_df=merge_df_by_column_name("Close","2021-01-06","2026-04-10",*port_list)
mult_df.corr()

price_plot=(mult_df/mult_df.iloc[0]*100).plot(figsize=(16,9))

returns=np.log(mult_df/mult_df.shift(1))
mean_return=returns.mean()*252
print(mean_return)
stdevs=returns.std()
print(stdevs)
corr=returns.corr()
print(corr)

In [52]:
p_ret=[]
p_vol=[]
p_SR=[]
p_wt=[]
for x in range(10000):
  p_weights=np.random.random(len(port_list))
  p_weights/=np.sum(p_weights)

  ret_1=np.sum(p_weights*returns.mean())*252
  p_ret.append(ret_1)

  vol_1=np.sqrt(np.dot(p_weights.T,np.dot(returns.cov()*252, p_weights)))
  p_vol.append(vol_1)

  SR_1=(ret_1-risk_free_rate)/vol_1
  p_SR.append(SR_1)

  p_wt.append(p_weights)

p_ret=np.array(p_ret)
p_vol=np.array(p_vol)
p_SR=np.array(p_SR)
p_wt=np.array(p_wt)

In [ ]:
ports=pd.DataFrame({'Return':p_ret,'Volatility':p_vol})
#Plot efficient frontier
ports.plot(x='Volatility',y='Return',kind='scatter',figsize=(16,9))

In [ ]:
SR_idx=np.argmax(p_SR)
i=0
while i<len(port_list):
  print("Stock :%s:%2.2f"%(port_list[i],(p_wt[SR_idx][i]*100)))
  i+=1
print("\nVolatility: ",p_vol[SR_idx])
print("\nReturn: ",p_ret[SR_idx])